In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset

class CandleDataset(Dataset):
    def __init__(self, df, feature_cols, window=60):
        self.features = df[feature_cols].values.astype(np.float32)
        self.labels = df["label"].values.astype(np.int64)
        self.window = window

    def __len__(self):
        return max(0, len(self.features) - self.window)

    def __getitem__(self, idx):
        x = self.features[idx : idx + self.window]          # (window, n_features)
        y = self.labels[idx + self.window - 1]                # лейбл на конец окна
        return torch.tensor(x), torch.tensor(y)

In [ ]:
from model import prepare_features_and_labels

class WindowExampleDataset(Dataset):
    """Каждый df целиком = один пример: window строк фичей -> 1 label"""
    def __init__(self, dfs, feature_cols, window=120, horizon=5, flat_threshold=0.001):
        self.examples = []
        
        for df in dfs:
            
            if len(df) < window:
                print(len(df))
                continue 
            
            context = df.iloc[:window]
            #print(len(df))
            close_at_context_end = df["close"].iloc[window - 1]
            close_at_horizon_end = df["close"].iloc[window + horizon - 1]
            future_return = close_at_horizon_end - close_at_context_end
            
            if future_return > flat_threshold:
                label = 2
            elif future_return < -flat_threshold:
                label = 0
            else:
                label = 1
            
            x = context[feature_cols].values.astype(np.float32)
            self.examples.append((x, label))
        
        print(f"Valid examples: {len(self.examples)} / {len(dfs)}")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        x, y = self.examples[idx]
        return torch.tensor(x), torch.tensor(y, dtype=torch.int64)

In [ ]:
print(torch.__version__)  
print(torch.cuda.is_available())

In [ ]:
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score
from model import LSTMClassifier
import torch.nn as nn
import os

def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3, device="cuda", checkpoint_dir="models/checkpoints"):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    best_val_loss = float("inf")
    epochs_without_improvement = 0
    history = []

    for epoch in range(epochs):
        # --- training ---
        model.train()
        train_loss = 0
        for x, y in train_loader: 
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        # --- validation ---
        model.eval()
        val_loss = 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss = criterion(logits, y)   # same criterion, now on val data too
                val_loss += loss.item()
                
                preds = logits.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(y.cpu().numpy())
            val_loss /= len(val_loader) # same as train_loss

        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average="macro")
        print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={acc:.4f}, val_f1={f1:.4f}")
        
        
        history.append({
            "epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss,
            "val_acc": acc, "val_f1": f1
        })
        
        # --- save the BEST checkpoint so far (lowest val_loss) ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "val_acc": acc,
                "val_f1": f1,
            }, f"{checkpoint_dir}/best.pt")
        else:
            epochs_without_improvement += 1

        # --- optional: periodic checkpoint every N epochs, for later inspection ---
        if (epoch + 1) % 50 == 0:
            torch.save(model.state_dict(), f"{checkpoint_dir}/epoch_{epoch+1}.pt")
            
    # always end by restoring the best checkpoint, not the last epoch
    best_checkpoint = torch.load(f"{checkpoint_dir}/best.pt", map_location=device)
    model.load_state_dict(best_checkpoint["model_state_dict"])
    print(f"Loaded best model from epoch {best_checkpoint['epoch']} "
        f"(val_loss={best_checkpoint['val_loss']:.4f})")
        

    return model, history

In [ ]:
import pandas as pd
from generator import loadDataSet
from model import prepare_features_and_labels

# loading train data
dfs_raw = loadDataSet('data/synthetic/0046_002_4_5')

dfs = [0]*len(dfs_raw)
print(len(dfs_raw))
for i,df in enumerate(dfs_raw):
    dfs[i] = prepare_features_and_labels(df, horizon=5, flat_threshold=0.001)
    


In [ ]:
import pandas as pd
from model import prepare_features_and_labels, feature_cols



# train/val split
split_idx = int(len(dfs) * 0.8)
train_dfs, val_dfs = dfs[:split_idx], dfs[split_idx:]


train_ds = WindowExampleDataset(train_dfs, feature_cols, window=120, horizon=5, flat_threshold=0.001)
val_ds = WindowExampleDataset(val_dfs, feature_cols, window=120, horizon=5, flat_threshold=0.001)


train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=True)

model = LSTMClassifier(n_features=len(feature_cols), num_layers=2, num_classes=3)
print(f"Learning on {'cuda' if torch.cuda.is_available() else 'cpu'}")
model, history = train_model(
    model, 
    train_loader, 
    val_loader,
    epochs=1,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# plot train/val loss
import matplotlib.pyplot as plt
df_hist = pd.DataFrame(history)

plt.plot(df_hist["epoch"], df_hist["train_loss"], label="train_loss")
plt.plot(df_hist["epoch"], df_hist["val_loss"], label="val_loss")
plt.legend()

In [ ]:
torch.save(model.state_dict(), "models/lstm_3360000_1_0046_23.pt")